In [10]:
# Configuracion
N_ITER = 4
CV = 2
RANDOM_STATE = 42
TARGET_CLASS = None
BASE_NAME = None
BALANCE_NAME = None


In [11]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

REPO_ROOT = Path('.').resolve()
INPUT_BASE = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion'
PRE_REPORTS_BASE = REPO_ROOT / 'reports' / '08_modelo_xgboost'
OUTPUT_BASE = REPO_ROOT / 'reports' / '13_tuning_xgboost'

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'UNDER': RandomUnderSampler(random_state=RANDOM_STATE),
    'SMOTEENN': SMOTEENN(random_state=RANDOM_STATE),
}


def latest_file(pattern_base: Path, pattern: str) -> Path:
    files = sorted(pattern_base.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No se encontro archivo con patron: {pattern_base / pattern}')
    return files[-1]


def latest_input_path() -> Path:
    candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f'No hay subdirectorios en {INPUT_BASE}')
    return candidates[-1]


def choose_candidates(summary_csv: Path) -> list[dict]:
    df = pd.read_csv(summary_csv)
    req = {'modelo', 'balanceo', 'cv_recall_target', 'cv_macro_f1'}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f'Faltan columnas en resumen: {sorted(miss)}')

    d = df.dropna(subset=['cv_recall_target', 'cv_macro_f1']).copy()
    if d.empty:
        raise ValueError('No hay filas validas con cv_recall_target y cv_macro_f1')

    by_recall = d.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]
    by_general = d.sort_values(['cv_macro_f1', 'cv_recall_target'], ascending=False).iloc[0]

    candidates = [
        {
            'tag': 'best_recall_target',
            'base_name': str(by_recall['modelo']),
            'balance_name': str(by_recall['balanceo']),
            'cv_recall_target': float(by_recall['cv_recall_target']),
            'cv_macro_f1': float(by_recall['cv_macro_f1']),
        },
        {
            'tag': 'best_general',
            'base_name': str(by_general['modelo']),
            'balance_name': str(by_general['balanceo']),
            'cv_recall_target': float(by_general['cv_recall_target']),
            'cv_macro_f1': float(by_general['cv_macro_f1']),
        },
    ]

    dedup = []
    seen = set()
    for c in candidates:
        key = (c['base_name'], c['balance_name'])
        if key in seen:
            continue
        seen.add(key)
        dedup.append(c)
    return dedup


def resolve_target_class(y: pd.Series, target: int | None) -> int:
    classes = list(pd.Series(y).unique())
    if target is None:
        return int(pd.Series(y).min())
    if target in classes:
        return int(target)
    return int(classes[0])


def build_pipeline(balancer) -> Pipeline:
    model = XGBClassifier(
        objective='multi:softprob',
        use_label_encoder=False,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    steps = []
    if balancer is not None:
        steps.append(('balance', clone(balancer)))
    steps.append(('model', model))
    return Pipeline(steps)


def save_eval_pdf(y_true, y_pred, y_proba, class_order, out_pdf: Path, title_prefix: str):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=class_order)

    with PdfPages(out_pdf) as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap='Blues')
        plt.title(f'{title_prefix} - Matriz de Confusion')
        pdf.savefig(); plt.close()

        if np.unique(y_true).size >= 2:
            y_bin = label_binarize(y_true, classes=class_order)
            proba_for_curves = y_proba

            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
                if proba_for_curves.shape[1] == 1:
                    proba_for_curves = np.hstack([1 - proba_for_curves, proba_for_curves])

            plt.figure()
            for i, clase in enumerate(class_order):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title(f'{title_prefix} - Curvas ROC por clase')
            plt.xlabel('FPR')
            plt.ylabel('TPR')
            plt.legend()
            pdf.savefig(); plt.close()

            plt.figure()
            for i, clase in enumerate(class_order):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
            plt.title(f'{title_prefix} - Curvas Precision-Recall por clase')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.legend()
            pdf.savefig(); plt.close()


In [12]:
summary_csv = latest_file(PRE_REPORTS_BASE, '**/resumen_modelos_xgboost.csv')
input_path = latest_input_path()

if BASE_NAME and BALANCE_NAME:
    candidates = [{
        'tag': 'manual',
        'base_name': BASE_NAME,
        'balance_name': BALANCE_NAME,
        'cv_recall_target': None,
        'cv_macro_f1': None,
    }]
else:
    candidates = choose_candidates(summary_csv)

run_id = datetime.today().strftime('%Y%m%d')
output_dir = OUTPUT_BASE / run_id / f"XGBoost_tuning_{datetime.today().strftime('%Y-%m-%d')}"
output_dir.mkdir(parents=True, exist_ok=True)

all_summary = []

for cand in candidates:
    tag = cand['tag']
    base_name = cand['base_name']
    balance_name = cand['balance_name']

    if balance_name not in BALANCE_METHODS:
        raise ValueError(f'Balanceo no soportado: {balance_name}')

    print(f"\n=== Candidato: {tag} | {base_name} [{balance_name}] ===")

    X_train = pd.read_parquet(input_path / f'X_train_{base_name}.parquet')
    X_test = pd.read_parquet(input_path / f'X_test_{base_name}.parquet')
    X_train = X_train.astype('float32')
    X_test = X_test.astype('float32')
    y_train = pd.read_parquet(input_path / f'y_train_{base_name}.parquet').squeeze()
    y_test = pd.read_parquet(input_path / f'y_test_{base_name}.parquet').squeeze()

    target_class = resolve_target_class(y_train, TARGET_CLASS)
    y_min = int(y_train.min())
    y_train_adj = y_train - y_min
    y_test_adj = y_test - y_min
    target_class_adj = target_class - y_min

    balancer = BALANCE_METHODS[balance_name]
    pipeline = build_pipeline(balancer)

    scorers = {
        'recall_target': make_scorer(recall_score, labels=[target_class_adj], average='macro', zero_division=0),
        'f1_macro': 'f1_macro',
    }

    param_distributions = {
        'model__n_estimators': [120, 180],
        'model__max_depth': [4, 6],
        'model__learning_rate': [0.05, 0.08],
        'model__subsample': [0.85, 1.0],
        'model__colsample_bytree': [0.85, 1.0],
        'model__min_child_weight': [1, 3],
        'model__gamma': [0.0, 0.1],
        'model__reg_alpha': [0.0, 0.1],
        'model__reg_lambda': [1.0, 2.0],
    }

    cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring=scorers,
        refit='recall_target',
        cv=cv,
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=1,
        return_train_score=False,
    )
    search.fit(X_train, y_train_adj)

    best_model = search.best_estimator_
    y_pred_test = best_model.predict(X_test) + y_min
    y_pred_train = best_model.predict(X_train) + y_min
    y_proba = best_model.predict_proba(X_test)
    class_order = best_model.named_steps['model'].classes_ + y_min

    cand_dir = output_dir / tag
    cand_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = cand_dir / 'reporte_tuning_xgboost.pdf'
    save_eval_pdf(y_test, y_pred_test, y_proba, class_order, pdf_path, f'{tag} - {base_name} [{balance_name}]')

    cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_recall_target')
    cv_results.to_csv(cand_dir / 'cv_results_xgboost_tuning.csv', index=False)

    summary = {
        'candidate_tag': tag,
        'base_name': base_name,
        'balanceo': balance_name,
        'summary_csv_source': str(summary_csv),
        'input_path': str(input_path),
        'target_class_original': int(target_class),
        'selected_cv_recall_target': cand['cv_recall_target'],
        'selected_cv_macro_f1': cand['cv_macro_f1'],
        'cv_best_recall_target': float(search.best_score_),
        'cv_best_f1_macro': float(cv_results.iloc[0]['mean_test_f1_macro']),
        'test_accuracy': float(accuracy_score(y_test, y_pred_test)),
        'test_f1_macro': float(f1_score(y_test, y_pred_test, average='macro', zero_division=0)),
        'train_f1_macro': float(f1_score(y_train, y_pred_train, average='macro', zero_division=0)),
        'pdf_report': str(pdf_path),
    }

    pd.DataFrame([summary]).to_csv(cand_dir / 'resumen_tuning_xgboost.csv', index=False)
    with open(cand_dir / 'best_params_xgboost.json', 'w', encoding='utf-8') as f:
        json.dump(search.best_params_, f, indent=2)

    report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
    report_test.to_csv(cand_dir / 'classification_report_test_xgboost.csv')

    all_summary.append(summary)
    print(json.dumps(summary, indent=2))

pd.DataFrame(all_summary).to_csv(output_dir / 'resumen_tuning_xgboost_all_candidates.csv', index=False)
print(f'Salida: {output_dir}')



=== Candidato: best_recall_target | Robust_ORIGINAL_ALL [SMOTEENN] ===
Fitting 2 folds for each of 4 candidates, totalling 8 fits


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [02:56:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [03:24:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [03:53:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\tr

{
  "candidate_tag": "best_recall_target",
  "base_name": "Robust_ORIGINAL_ALL",
  "balanceo": "SMOTEENN",
  "summary_csv_source": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\08_modelo_xgboost\\20260210\\XGBoost_04_2026-02-10\\resumen_modelos_xgboost.csv",
  "input_path": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\data\\intermediate\\05_seleccion\\04_2026_01_12",
  "target_class_original": 1,
  "selected_cv_recall_target": 0.9008963423449472,
  "selected_cv_macro_f1": 0.266330242058423,
  "cv_best_recall_target": 0.9310394679774469,
  "cv_best_f1_macro": 0.23032825466410928,
  "test_accuracy": 0.27312708522899604,
  "test_f1_macro": 0.23464326693709064,
  "train_f1_macro": 0.23711031938829769,
  "pdf_report": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\13_tuning_xgboost\\20260309\\XGBoost_tuning_2026-03-09\\best_recall_target\\reporte_tuning_xgboost.pdf"
}

=== Candidato: best_general | 

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [08:11:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [08:28:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [08:46:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\tr

{
  "candidate_tag": "best_general",
  "base_name": "MinMax_FE_PCAopt",
  "balanceo": "SMOTE",
  "summary_csv_source": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\08_modelo_xgboost\\20260210\\XGBoost_04_2026-02-10\\resumen_modelos_xgboost.csv",
  "input_path": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\data\\intermediate\\05_seleccion\\04_2026_01_12",
  "target_class_original": 1,
  "selected_cv_recall_target": 0.2439882415305286,
  "selected_cv_macro_f1": 0.4202280014168591,
  "cv_best_recall_target": 0.26777022794082217,
  "cv_best_f1_macro": 0.4045217220212963,
  "test_accuracy": 0.4677157487198701,
  "test_f1_macro": 0.4052060714784079,
  "train_f1_macro": 0.42122270463375705,
  "pdf_report": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\13_tuning_xgboost\\20260309\\XGBoost_tuning_2026-03-09\\best_general\\reporte_tuning_xgboost.pdf"
}
Salida: C:\Users\Gonzalo\Documents\GitHub\TesisAus